# DE-Har 2025 — all-streams export

Single-CSV export of every in-situ and satellite observation at its **native resolution** for 2025, joined on the datetime index. Design rules:

- **No filters** — no rain, humidity, manual clear-sky or any other drop. Every scene, every observation.
- **One column for PAI** — total canopy PAI per scan (cumulative `LinearPAI` at the top of the profile).
- **One column for soil moisture** — mean across all profile × depth VWC columns (no IQR guard).
- **All other streams kept per tree / receiver / camera** (sap flow, TWD, SWP per tree; VOD per receiver; leaf angle per cam).
- **Sentinel-1 / Sentinel-2** reuse the `time_series.ipynb` processing: 100 m ROI mean + std at the tower, spyndex indices for S2, cosine-θ-normalised dB indices for S1 (both ascending + descending, prefixed).
- All timestamps **UTC-aware**.
- Output: `data/processed/dehar_all_streams_2025.csv`.

In [1]:
from __future__ import annotations
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import spyndex

# Config ────────────────────────────────────────────────────────────────────
TARGET_YEAR      = 2025
TOWER_X, TOWER_Y = 395509.71, 5309956.22          # EPSG:32632 (UTM 32N)
ROI_RADIUS_M     = 100.0

S2_INDICES       = ["NDVI", "kNDVI", "EVI", "NIRv", "SAVI", "CIRE", "MTCI", "NDII"]
S2_BANDS_KEEP    = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]
S1_LINEAR_VARS   = ["VV_lin","VH_lin","SPAN_lin","CR_lin","RVI"]
S1_THETA_REF     = 40.0
S1_N             = 2

# Paths ─────────────────────────────────────────────────────────────────────
ROOT_DIR = Path.cwd().parents[2]
DATA_DIR = ROOT_DIR / "data"
OUT_PATH = DATA_DIR / "processed" / "dehar_all_streams_2025.csv"

P = {
    "meteo":      DATA_DIR / "processed/atmosphere_soil/meteo_dehar_30min.csv",
    "fluxes":     DATA_DIR / "processed/atmosphere_soil/fluxes_dehar_30min.csv",
    "sm":         DATA_DIR / "processed/atmosphere_soil/soil_moisture_dehar_30min.csv",
    "precip":     DATA_DIR / "processed/atmosphere_soil/HARTHM_2025_Precipitation_30min_UTC.csv",
    "sapflow":    DATA_DIR / "processed/physiology/sap_flux_density/sapflow_dehar_30min.csv",
    "swp":        DATA_DIR / "processed/physiology/stemwater_potential/swp_dehar_15min.csv",
    "twd":        DATA_DIR / "processed/physiology/twd/twd_dehar_30min.csv",
    "vod":        DATA_DIR / "processed/proximal_rs/gnss_vod/gnss_vod_dehar_30min.csv",
    "leaf_angle": DATA_DIR / "processed/proximal_rs/anglecam/leaf_angle_cam60_70_native.csv",
    "pai":        DATA_DIR / "processed/proximal_rs/leaf/leaf_hemi_hi_2025_scan05utc.csv",
    "s1_dir":     DATA_DIR / "processed/satellite/sentinel1",
    "s2_dir":     DATA_DIR / "processed/satellite/sentinel2",
}

print(f"Target year: {TARGET_YEAR}  |  Output: {OUT_PATH}")

Target year: 2025  |  Output: /mnt/data/lk1167/projects/dehar-spac/data/processed/dehar_all_streams_2025.csv


## 1. Load in-situ streams (2025, UTC-aware, native resolution)

In [2]:
def load_csv(path: Path, datetime_col: str = "datetime") -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=[datetime_col], index_col=datetime_col)
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    return df[df.index.year == TARGET_YEAR].copy()

meteo      = load_csv(P["meteo"])
fluxes     = load_csv(P["fluxes"])
sm_raw     = load_csv(P["sm"])
precip     = load_csv(P["precip"], datetime_col="From")[["Precipitation_Sum_mm"]]
precip     = precip.rename(columns={"Precipitation_Sum_mm": "precip_mm"})
precip.index.name = "datetime"
sapflow    = load_csv(P["sapflow"])
swp        = load_csv(P["swp"])          # 15-min
twd        = load_csv(P["twd"])
vod        = load_csv(P["vod"])
leaf_angle = load_csv(P["leaf_angle"])   # all 11 cams (60-70)

for name, df in [("meteo",meteo),("fluxes",fluxes),("sm_raw",sm_raw),("precip",precip),
                 ("sapflow",sapflow),("swp",swp),("twd",twd),("vod",vod),
                 ("leaf_angle",leaf_angle)]:
    print(f"{name:12s} {df.index.min()} → {df.index.max()}   n={len(df):>9,}   cols={len(df.columns)}")

meteo        2025-01-01 00:00:00+00:00 → 2025-11-30 23:00:00+00:00   n=   16,031   cols=6
fluxes       2025-01-01 00:00:00+00:00 → 2025-11-30 23:00:00+00:00   n=   16,031   cols=10
sm_raw       2025-01-01 00:00:00+00:00 → 2025-11-30 23:00:00+00:00   n=   16,031   cols=8
precip       2025-01-01 00:00:00+00:00 → 2025-12-31 23:30:00+00:00   n=   17,520   cols=1
sapflow      2025-01-01 00:00:00+00:00 → 2025-11-21 10:00:00+00:00   n=   15,576   cols=5
swp          2025-05-05 23:15:00+00:00 → 2025-11-21 10:15:00+00:00   n=   19,148   cols=3
twd          2025-01-01 00:00:00+00:00 → 2025-11-21 10:00:00+00:00   n=   15,558   cols=6
vod          2025-04-17 14:30:00+00:00 → 2025-12-14 23:00:00+00:00   n=    9,480   cols=3
leaf_angle   2025-03-26 06:39:01+00:00 → 2025-12-02 15:43:27+00:00   n=1,633,411   cols=11


## 2. Collapse soil moisture and PAI to a single column each

In [3]:
# Soil moisture — mean per profile (depth average), then plot mean ± std ────
sm_cols = [c for c in sm_raw.columns if c.startswith("vwc_pct_")]

# Extract unique profiles (a, b, c)
import re
profiles = sorted({re.match(r"vwc_pct_([a-z]+)_", c).group(1)
                   for c in sm_cols if re.match(r"vwc_pct_([a-z]+)_", c)})

# Step 1: depth-average within each profile
profile_means = pd.DataFrame({
    f"sm_pct_{p}": sm_raw[[c for c in sm_cols if c.startswith(f"vwc_pct_{p}_")]].mean(axis=1, skipna=True)
    for p in profiles
})

# Step 2: plot-level mean and std across profiles
sm = pd.DataFrame({
    "sm_mean_pct": profile_means.mean(axis=1, skipna=True),
    "sm_std_pct":  profile_means.std(axis=1, skipna=True),
})

print(f"Soil moisture: {len(sm_cols)} raw columns → {len(profiles)} profiles ({profiles}) → plot mean ± std")
print(sm_cols)

# PAI — total canopy PAI per scan (cumulative LinearPAI at the top of the profile)
pai_long = pd.read_csv(P["pai"], parse_dates=["datetime"])
pai_long["datetime"] = pd.to_datetime(pai_long["datetime"], utc=True)
pai_long = pai_long[pai_long["datetime"].dt.year == TARGET_YEAR]
pai = (pai_long.groupby("datetime")["LinearPAI"].max()
                .to_frame("pai_total")
                .sort_index())
print(f"PAI: reduced {pai_long['datetime'].nunique()} scans × {pai_long['height'].nunique()} height-layers "
      f"→ single 'pai_total' column ({len(pai)} rows)")

Soil moisture: 8 raw columns → 3 profiles (['a', 'b', 'c']) → plot mean ± std
['vwc_pct_a_5cm', 'vwc_pct_a_10cm', 'vwc_pct_a_20cm', 'vwc_pct_b_5cm', 'vwc_pct_b_10cm', 'vwc_pct_b_20cm', 'vwc_pct_c_5cm', 'vwc_pct_c_10cm']
PAI: reduced 228 scans × 50 height-layers → single 'pai_total' column (228 rows)


## 3. Sentinel-2 — 100 m ROI mean + std, all spyndex indices + raw bands, **no scene filter**

In [4]:
## 3. Load pre-filtered Sentinel-2
s2 = pd.read_csv(
    DATA_DIR / "processed/satellite/sentinel2_filtered_2025.csv",
    index_col="time", parse_dates=True,
)
s2.index = pd.DatetimeIndex(s2.index).tz_localize("UTC")
s2.index.name = "datetime"
s2 = s2.drop(columns=["interpolated"], errors="ignore")
s2 = s2.add_prefix("s2_")
print(f"Sentinel-2: {len(s2)} scenes, {len(s2.columns)} columns")

Sentinel-2: 46 scenes, 32 columns


In [ ]:
# old approach. look at /mnt/data/lk1167/projects/dehar-spac/notebooks/01_processing/sentinel2_filter_dehar.ipynb for new processing.
# dates_to_consider = [
#     "2025-01-13", "2025-02-27", "2025-03-04", "2025-03-09", "2025-03-19",
#     "2025-04-03", "2025-04-08", "2025-04-10", "2025-04-28", "2025-04-30",
#     "2025-05-10", "2025-05-20", "2025-05-30", "2025-06-09", "2025-06-12",
#     "2025-06-17", "2025-06-19", "2025-06-22", "2025-06-29", "2025-07-02",
#     "2025-08-08", "2025-08-11", "2025-08-18", "2025-08-26", "2025-08-31",
#     "2025-09-20", "2025-10-07", "2025-10-15", "2025-10-30", "2025-11-04",
#     "2025-11-06", "2025-11-19", "2025-12-09",
# ]

# def _norm_dims(ds):
#     rn = {}
#     if "X" in ds.dims and "x" not in ds.dims: rn["X"] = "x"
#     if "Y" in ds.dims and "y" not in ds.dims: rn["Y"] = "y"
#     return ds.rename(rn) if rn else ds

# def load_s2_indices(data_dir: Path) -> xr.Dataset:
#     files = sorted(Path(data_dir).glob("s2_sr_dehar_*_indices.nc"))
#     if not files: raise FileNotFoundError(data_dir)
#     ds = xr.concat([_norm_dims(xr.open_dataset(f)) for f in files],
#                    dim="time").sortby("time")
#     _, idx = np.unique(ds.time.values, return_index=True)
#     return ds.isel(time=idx)

# def add_spyndex_indices(ds, indices=S2_INDICES):
#     ds = ds / 10000.0 if ds["B2"].max() > 2.0 else ds
#     kNR = spyndex.computeKernel(kernel="RBF", params={
#         "a": ds["B8"].astype("float32"), "b": ds["B4"].astype("float32"),
#         "sigma": float(np.abs(ds["B8"] - ds["B4"]).median().item()),
#     })
#     band_params = {
#         "N": ds["B8"].astype("float32"),  "N2": ds["B8A"].astype("float32"),
#         "R": ds["B4"].astype("float32"),  "G":  ds["B3"].astype("float32"),
#         "B": ds["B2"].astype("float32"),  "RE1":ds["B5"].astype("float32"),
#         "RE2":ds["B6"].astype("float32"), "S1": ds["B11"].astype("float32"),
#         "S2": ds["B12"].astype("float32"),"kNN":1.0, "kNR":kNR,
#     }
#     for name in indices:
#         params = dict(band_params)
#         for p in spyndex.indices[name].bands:
#             if p not in params and p in spyndex.constants:
#                 params[p] = spyndex.constants[p].default
#         ds[name] = spyndex.computeIndex(index=name, params=params, online=False)
#     return ds

# def s2_roi_stats(data_dir, year, x=TOWER_X, y=TOWER_Y, radius_m=ROI_RADIUS_M,
#                  indices=S2_INDICES, bands=S2_BANDS_KEEP, dates=None) -> pd.DataFrame:
#     ds = load_s2_indices(data_dir)
#     ds = add_spyndex_indices(ds, indices=indices)
#     ds_y = ds.sel(time=ds.time.dt.year == year)
#     mask = ((ds_y.x - x) ** 2 + (ds_y.y - y) ** 2) <= radius_m ** 2
#     keep = [v for v in ds_y.data_vars
#             if {"time","x","y"}.issubset(set(ds_y[v].dims)) and v in (list(indices) + list(bands))]
#     roi = ds_y[keep].where(mask)
#     out = roi.mean(dim=("x","y"), skipna=True).to_dataframe().add_suffix("_mean")
#     out = out.join(roi.std(dim=("x","y"), skipna=True).to_dataframe().add_suffix("_std"))
#     out = out.sort_index()
#     if out.index.tz is None:
#         out.index = pd.DatetimeIndex(out.index).tz_localize("UTC")
#     out.index.name = "datetime"
#     out = out.add_prefix("s2_")        # e.g. s2_NDVI_mean
#     if dates is not None:
#         date_set = {pd.Timestamp(d).date() for d in dates}
#         out = out[np.isin(out.index.date, list(date_set))]
#     return out

# s2 = s2_roi_stats(P["s2_dir"], TARGET_YEAR, dates=dates_to_consider)
# print(f"Sentinel-2: {len(s2)} scenes kept (date-filtered), {len(s2.columns)} columns")
# s2.head(5)

Sentinel-2: 33 scenes kept (date-filtered), 36 columns


,s2_B2_mean,s2_B3_mean,s2_B4_mean,s2_B5_mean,s2_B6_mean,s2_B7_mean,s2_B8_mean,s2_B8A_mean,s2_B11_mean,s2_B12_mean,...,s2_B11_std,s2_B12_std,s2_NDVI_std,s2_EVI_std,s2_kNDVI_std,s2_NIRv_std,s2_SAVI_std,s2_CIRE_std,s2_MTCI_std,s2_NDII_std
datetime,,,,,,,,,,,,,,,,,,,,,
2025-01-13 10:37:34.949000+00:00,0.012106,0.021834,0.026403,0.049577,0.097620,0.114146,0.136689,0.135597,0.092822,0.055390,...,0.020799,0.014661,0.112244,0.058799,0.085197,0.028909,0.048107,0.673099,2.174922,0.120664
2025-02-27 10:37:59.976000+00:00,0.051701,0.058832,0.058221,0.089582,0.135409,0.149796,0.175390,0.174833,0.152337,0.087789,...,0.043324,0.028157,0.078470,0.049356,0.078996,0.021640,0.037177,0.265004,1.522062,0.083783
2025-03-04 10:37:38.912000+00:00,0.039170,0.052499,0.058044,0.087339,0.135294,0.154393,0.176539,0.179112,0.176835,0.114017,...,0.035475,0.027408,0.080453,0.037265,0.055304,0.016760,0.027929,0.295458,1.578844,0.080779
2025-03-09 10:37:59.402000+00:00,0.028674,0.046437,0.052236,0.084441,0.129996,0.144922,0.165416,0.170644,0.175823,0.106703,...,0.039310,0.029633,0.090954,0.036154,0.052753,0.016974,0.028367,0.301903,0.967061,0.088193
2025-03-19 10:37:59.665000+00:00,0.024604,0.041827,0.049489,0.082022,0.127773,0.143762,0.167136,0.169433,0.174829,0.105988,...,0.037802,0.027939,0.093512,0.034016,0.053186,0.016679,0.027274,0.320846,0.949141,0.084511


## 4. Sentinel-1 — cosine-θ-normalised dB indices at 100 m ROI, both orbits, **no filter**

In [5]:
## 4. Load pre-filtered Sentinel-1
s1 = pd.read_csv(
    DATA_DIR / "processed/satellite/sentinel1_filtered_2025.csv",
    index_col="time", parse_dates=True,
)
s1.index = pd.DatetimeIndex(s1.index).tz_localize("UTC")
s1.index.name = "datetime"
s1 = s1.add_prefix("s1_")
print(f"Sentinel-1: {len(s1)} scenes, {len(s1.columns)} columns")

Sentinel-1: 44 scenes, 16 columns


In [ ]:
# old approach. look at /mnt/data/lk1167/projects/dehar-spac/notebooks/01_processing/sentinel1_filter_dehar.ipynb for new processing.
# def _db_to_lin(da): return 10 ** (da / 10.0)

# def _angle_normalise(vv_db, vh_db, angle_deg, theta_ref=S1_THETA_REF, n=S1_N):
#     theta   = np.deg2rad(angle_deg)
#     theta_r = np.deg2rad(theta_ref)
#     corr_db = 10 * n * np.log10(np.cos(theta_r) / np.cos(theta))
#     return vv_db + corr_db, vh_db + corr_db

# def _add_s1_linear(ds):
#     vv_lin = _db_to_lin(ds["VV"]); vh_lin = _db_to_lin(ds["VH"])
#     ds["VV_lin"]   = vv_lin
#     ds["VH_lin"]   = vh_lin
#     ds["SPAN_lin"] = vv_lin + vh_lin
#     ds["CR_lin"]   = vh_lin / vv_lin.where(vv_lin > 0)
#     ds["RVI"]      = 4 * vh_lin / (vv_lin + vh_lin)
#     return ds

# def load_s1(data_dir, year, orbit_letter):
#     f = sorted(Path(data_dir).glob(f"s1_grd_dehar_{orbit_letter}_{year}*.nc"))
#     if not f: raise FileNotFoundError(orbit_letter)
#     ds = xr.open_dataset(f[0])
#     rn = {k: k.lower() for k in ds.dims if k in ("X","Y")}
#     return ds.rename(rn) if rn else ds

# def s1_roi_stats(data_dir, year, orbit_letter, x=TOWER_X, y=TOWER_Y,
#                  radius_m=ROI_RADIUS_M) -> pd.DataFrame:
#     ds = load_s1(data_dir, year, orbit_letter)
#     ds["VV"], ds["VH"] = _angle_normalise(ds["VV"], ds["VH"], ds["angle"])
#     ds = _add_s1_linear(ds)
#     ds_y = ds.sel(time=ds.time.dt.year == year)
#     mask = ((ds_y.x - x) ** 2 + (ds_y.y - y) ** 2) <= radius_m ** 2
#     keep = [v for v in ds_y.data_vars
#             if {"time","x","y"}.issubset(set(ds_y[v].dims))
#             and v in (S1_LINEAR_VARS + ["angle"])]
#     roi = ds_y[keep].where(mask)
#     df = roi.mean(dim=("x","y"), skipna=True).to_dataframe().add_suffix("_mean")
#     df = df.join(roi.std(dim=("x","y"), skipna=True).to_dataframe().add_suffix("_std"))
#     df = df.sort_index()
#     if df.index.tz is None:
#         df.index = pd.DatetimeIndex(df.index).tz_localize("UTC")
#     df.index.name = "datetime"
#     # linear → dB display, with propagated std
#     k = 10 / np.log(10)
#     for v in ["VV","VH","SPAN","CR"]:
#         df[f"{v}_dB_mean"] = 10 * np.log10(df[f"{v}_lin_mean"])
#         df[f"{v}_dB_std"]  = k * df[f"{v}_lin_std"] / df[f"{v}_lin_mean"]
#     prefix = "s1_asc_" if orbit_letter == "a" else "s1_desc_"
#     return df.add_prefix(prefix)

# s1_asc  = s1_roi_stats(P["s1_dir"], TARGET_YEAR, "a")
# s1_desc = s1_roi_stats(P["s1_dir"], TARGET_YEAR, "d")
# print(f"Sentinel-1 ascending  : {len(s1_asc)}  scenes × {len(s1_asc.columns)}  cols")
# print(f"Sentinel-1 descending : {len(s1_desc)} scenes × {len(s1_desc.columns)} cols")
# s1_asc.head(2)

Sentinel-1 ascending  : 51  scenes × 20  cols
Sentinel-1 descending : 49 scenes × 20 cols


,s1_asc_angle_mean,s1_asc_RVI_mean,s1_asc_VV_lin_mean,s1_asc_VH_lin_mean,s1_asc_SPAN_lin_mean,s1_asc_CR_lin_mean,s1_asc_angle_std,s1_asc_RVI_std,s1_asc_VV_lin_std,s1_asc_VH_lin_std,s1_asc_SPAN_lin_std,s1_asc_CR_lin_std,s1_asc_VV_dB_mean,s1_asc_VV_dB_std,s1_asc_VH_dB_mean,s1_asc_VH_dB_std,s1_asc_SPAN_dB_mean,s1_asc_SPAN_dB_std,s1_asc_CR_dB_mean,s1_asc_CR_dB_std
datetime,,,,,,,,,,,,,,,,,,,,
2025-01-10 17:24:09+00:00,41.690369,0.913288,0.102332,0.029482,0.131814,0.310337,0.002835,0.314938,0.032273,0.012308,0.037084,0.14355,-9.899879,1.369677,-15.304404,1.813024,-8.800374,1.221813,-5.081658,2.008871
2025-01-22 17:24:08+00:00,41.695316,0.811008,0.110631,0.027870,0.138501,0.262818,0.002836,0.251493,0.028346,0.010521,0.032151,0.10861,-9.561227,1.112761,-15.548686,1.639482,-8.585479,1.008152,-5.803457,1.794728


## 5. Phenology data

In [6]:
# Phenology — GCC from G5Bullet cameras (cam60–70), 2-min → 30-min mean ────
import re

PHENO_SCRATCH = DATA_DIR / "processed/proximal_rs/phenology/scratch"
PHENO_CAMS    = [f"G5Bullet_{i}" for i in range(60, 71)]   # _60 … _70

gcc_cam_frames = []

for cam in PHENO_CAMS:
    cam_num = re.search(r"_(\d+)$", cam).group(1)   # "60" … "70"
    col     = f"gcc_cam{cam_num}"

    files = sorted(PHENO_SCRATCH.glob(f"{cam}_*.parquet"))
    if not files:
        print(f"  {cam}: no scratch files found, skipping")
        continue

    # Concatenate all daily parquet files for this camera
    daily_frames = [pd.read_parquet(f) for f in files
                    if not pd.read_parquet(f).empty]
    if not daily_frames:
        print(f"  {cam}: all files empty, skipping")
        continue

    cam_df = pd.concat(daily_frames, ignore_index=True)

    # Timestamps are local time (Europe/Berlin = CET/CEST) → convert to UTC
    cam_df["datetime"] = (
        pd.to_datetime(cam_df["timestamp"])
          .dt.tz_localize("Europe/Berlin", ambiguous="infer",
                          nonexistent="shift_forward")
          .dt.tz_convert("UTC")
    )
    cam_df = (cam_df.drop(columns=["timestamp"])
                    .set_index("datetime")
                    .sort_index())
    cam_df = cam_df[cam_df.index.year == TARGET_YEAR]

    # 30-min mean GCC
    gcc_30min = cam_df["gcc"].resample("30min").mean().rename(col)
    gcc_cam_frames.append(gcc_30min)

    print(f"  {cam}: {len(gcc_30min.dropna()):>6,} valid 30-min steps  "
          f"({cam_df.index.min().date()} → {cam_df.index.max().date()})")

# Combine all cameras into wide DataFrame
phenology = pd.concat(gcc_cam_frames, axis=1).sort_index()
phenology.index.name = "datetime"
print(f"\nPhenology (GCC 30-min): {phenology.shape}  |  cols: {list(phenology.columns)}")

  G5Bullet_60:  5,951 valid 30-min steps  (2025-05-01 → 2025-11-30)
  G5Bullet_61:  6,039 valid 30-min steps  (2025-05-01 → 2025-11-30)
  G5Bullet_62:  6,085 valid 30-min steps  (2025-04-27 → 2025-11-30)
  G5Bullet_63:  6,122 valid 30-min steps  (2025-05-01 → 2025-11-30)
  G5Bullet_64:  6,028 valid 30-min steps  (2025-04-27 → 2025-11-30)
  G5Bullet_65:  5,940 valid 30-min steps  (2025-05-01 → 2025-11-30)
  G5Bullet_66:  6,079 valid 30-min steps  (2025-05-01 → 2025-11-30)
  G5Bullet_67:  6,150 valid 30-min steps  (2025-05-01 → 2025-12-02)
  G5Bullet_68:  6,064 valid 30-min steps  (2025-03-26 → 2025-11-30)
  G5Bullet_69:  6,049 valid 30-min steps  (2025-04-26 → 2025-12-02)
  G5Bullet_70:  6,091 valid 30-min steps  (2025-04-23 → 2025-11-30)

Phenology (GCC 30-min): (12067, 11)  |  cols: ['gcc_cam60', 'gcc_cam61', 'gcc_cam62', 'gcc_cam63', 'gcc_cam64', 'gcc_cam65', 'gcc_cam66', 'gcc_cam67', 'gcc_cam68', 'gcc_cam69', 'gcc_cam70']


## 6. Outer-join every stream on the datetime index and save

Every row is a single timestamp with one or more streams populated. Rows without data for a given stream carry `NaN` for those columns — that's by design, so the file preserves native resolution without resampling.

In [7]:
# Rename per-stream columns to avoid collisions / clarify provenance ────────
meteo_x   = meteo[["tair_c","rh_pct","vpd_hpa","rg_wm2","par_umol_m2s","ustar_ms"]]
fluxes_x  = fluxes.copy()
precip_x  = precip.copy()
sm_x      = sm.copy()
sapflow_x = sapflow.copy()              # columns like js_h10545, ...
swp_x     = swp.copy()                  # swp_mpa_h10545, ...
twd_x     = twd.copy()                  # twd_um_h10529, ...
vod_x     = vod.copy()                  # nvod_gps1, ...
la_x      = leaf_angle.copy()           # leaf_angle_cam60 ... cam70
pai_x     = pai.copy()                  # pai_total

# Outer join on datetime index ──────────────────────────────────────────────
all_streams = [meteo_x, fluxes_x, sm_x, precip_x,
               sapflow_x, swp_x, twd_x, vod_x, la_x, pai_x,
               phenology, s2, s1]

combined = pd.concat(all_streams, axis=1, join="outer").sort_index()
combined.index.name = "datetime"

print(f"Combined shape : {combined.shape}   "
      f"(rows = unique UTC timestamps, cols = union across streams)")
print(f"Date range     : {combined.index.min()} → {combined.index.max()}")
print(f"Columns ({len(combined.columns)}):")
for c in combined.columns: print(f"  - {c}")

Combined shape : (1658976, 107)   (rows = unique UTC timestamps, cols = union across streams)
Date range     : 2025-01-01 00:00:00+00:00 → 2025-12-31 23:30:00+00:00
Columns (107):
  - tair_c
  - rh_pct
  - vpd_hpa
  - rg_wm2
  - par_umol_m2s
  - ustar_ms
  - nee_umol_m2s
  - nee_f_umol_m2s
  - nee_f_sd_umol_m2s
  - gpp_f_umol_m2s
  - gpp_f_sd_umol_m2s
  - reco_f_umol_m2s
  - reco_f_sd_umol_m2s
  - le_wm2
  - h_wm2
  - et_f_mm_h
  - sm_mean_pct
  - sm_std_pct
  - precip_mm
  - js_h10545
  - js_h10546
  - js_h10547
  - js_h10559
  - js_h10560
  - swp_mpa_h10545
  - swp_mpa_h10546
  - swp_mpa_h10560
  - twd_um_h10529
  - twd_um_h10545
  - twd_um_h10546
  - twd_um_h10547
  - twd_um_h10559
  - twd_um_h10560
  - nvod_gps1
  - nvod_gps3
  - nvod_gps5
  - leaf_angle_cam60
  - leaf_angle_cam61
  - leaf_angle_cam62
  - leaf_angle_cam63
  - leaf_angle_cam64
  - leaf_angle_cam65
  - leaf_angle_cam66
  - leaf_angle_cam67
  - leaf_angle_cam68
  - leaf_angle_cam69
  - leaf_angle_cam70
  - pai_total
 

In [8]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUT_PATH)

size_mb = OUT_PATH.stat().st_size / 1e6
print(f"Saved → {OUT_PATH}")
print(f"Size          : {size_mb:.1f} MB")
print(f"Rows × cols   : {combined.shape[0]:,} × {combined.shape[1]}")
print(f"Non-null cells: {int(combined.notna().sum().sum()):,} "
      f"({100*combined.notna().sum().sum()/combined.size:.2f}% filled)")

Saved → /mnt/data/lk1167/projects/dehar-spac/data/processed/dehar_all_streams_2025.csv
Size          : 256.1 MB
Rows × cols   : 1,658,976 × 107
Non-null cells: 2,288,438 (1.29% filled)
